# SETUP

In [1]:
# Read libraries
import pandas as pd
import numpy as np
import os
import networkx as nx
from tqdm import tqdm



# ML libraries
from sklearn.metrics.pairwise import cosine_similarity
from ortools.graph.python import min_cost_flow

# Open file in read mode

taxonomy_path = os.path.join("data", "taxonomy.txt")
count_of_products_per_level1_path = os.path.join("data", "count_of_products_per_level1.csv")
data_path = os.path.join("data", "ensae_export_without_l1.parquet")
ground_truth_path = os.path.join("data", "ground_truth_level_1.parquet")

## Read categories files

In [2]:
# --------- Lire la taxonomy depuis un fichier txt ----------
# Assumons que le fichier s'appelle "taxonomy.txt"
# Format attendu : id_path <tab> category_path
df_taxonomy = pd.read_csv(taxonomy_path, sep='\t', header=None, names=['id_path', 'category_path'])

# Nettoyage
df_taxonomy['category_path'] = df_taxonomy['category_path'].str.strip()
df_taxonomy['id_path'] = df_taxonomy['id_path'].str.strip()

# Construire le graphe dirigé de la taxonomie
G = nx.DiGraph()
root = "ROOT"  # racine commune
G.add_node(root)

for path in df_taxonomy['category_path']:
    parts = [p.strip() for p in path.split(">")]
    if parts:  # relier le level_1 à la racine
        G.add_edge(root, parts[0])
    for i in range(len(parts)-1):
        parent = parts[i]
        child = parts[i+1]
        G.add_edge(parent, child)

# Identifier les level_1
level_1_nodes = [p.split(">")[0].strip() for p in df_taxonomy['category_path']]
level_1_nodes = list(set(level_1_nodes))

print("Level 1 categories:", level_1_nodes)

Level 1 categories: ['Real Estate', 'health & beauty', 'food, beverages & tobacco', 'Car Rental', 'Employment', 'office supplies', 'software', 'Hotels/Resorts', 'Dating', 'arts & entertainment', 'Services', 'Ground/Cruises/Packages', 'furniture', 'cameras & optics', 'toys & games', 'Communication', 'electronics', 'business & industrial', 'Airlines', 'Goods', 'Travel', 'religious & ceremonial', 'vehicles & parts', 'luggage & bags', 'Gaming/Gambling', 'sporting goods', 'hardware', 'apparel & accessories', 'media', 'home & garden', 'mature', 'Finance Services', 'animals & pet supplies', 'baby & toddler']


In [3]:
# Extract level 1 category counts

df_categories_count = pd.read_csv(count_of_products_per_level1_path)
df_categories_count

# Keep only level 1 categories present in both dataframes
common_level1 = set(level_1_nodes).intersection(set(df_categories_count['level_1_name']))
df_categories_count = df_categories_count[df_categories_count['level_1_name'].isin(common_level1)]

print(f" Number of level 1 categories in taxonomy: {len(level_1_nodes)}")
print(f" Number of level 1 categories in count_of_products_per_level1.csv: {len(df_categories_count)}")
print(f"Dropping {len(level_1_nodes) - len(common_level1)} level 1 categories from taxonomy that are not in count_of_products_per_level1.csv")

 Number of level 1 categories in taxonomy: 34
 Number of level 1 categories in count_of_products_per_level1.csv: 21
Dropping 13 level 1 categories from taxonomy that are not in count_of_products_per_level1.csv


In [4]:
print("Number of products : ", df_categories_count["count"].sum())

Number of products :  128253


## Read catalog and ground truth files

In [5]:
df_catalog = pd.read_parquet(data_path, engine='pyarrow')
df_ground_truth = pd.read_parquet(ground_truth_path, engine='pyarrow')

In [6]:
# Verfie that count_of_products_per_level1.csv contains the same number of products by category as the ground truth
series_ground_truth = df_ground_truth["level_1_name"].value_counts()
series_count_of_products = df_categories_count.set_index("level_1_name")["count"]

print("Comparing product counts between ground truth and count_of_products_per_level1.csv:")
for category in common_level1:
    count_gt = series_ground_truth.get(category, 0)
    count_count = series_count_of_products.get(category, 0)
    if count_gt != count_count:
        print(f"Category: {category} - Ground Truth Count: {count_gt}, count_of_products_per_level1.csv Count: {count_count}")
print("Comparison complete. Count_ground_truth and count_of_product have the same number of products by category")


Comparing product counts between ground truth and count_of_products_per_level1.csv:
Comparison complete. Count_ground_truth and count_of_product have the same number of products by category


## Preprocessing df_catalog

In [7]:
import pandas as pd
import re
from sklearn.preprocessing import LabelEncoder

def preprocess_for_nlp(df, title_col='title', desc_col='description', brand_col='brand', id_col='hashed_external_id', price_col='sale_price', category_col='level_1_category'):
    """
    Preprocess products DataFrame for Modern NLP (Transformers) & Tabular tasks:
    - Creates a 'Rich Text' string containing context (Title, Desc, Brand, Price)
    - Encodes brand as integer for tabular models (LightGBM/XGBoost)
    - Cleans text lightly (keeps punctuation, fixes spaces)
    - Keeps the Ground Truth category for fine-tuning
    """
    # Travailler sur une copie pour éviter les warnings de Pandas
    df = df.copy()

    # 1. Gérer les valeurs manquantes
    df[title_col] = df[title_col].fillna('')
    df[desc_col] = df[desc_col].fillna('')
    df[brand_col] = df[brand_col].fillna('Unknown')

    # 2. Sécuriser et nettoyer la colonne Prix
    if price_col in df.columns:
        # Convertir en numérique (transforme les erreurs ou textes bizarres en NaN)
        df[price_col] = pd.to_numeric(df[price_col], errors='coerce')
        # Remplacer les NaN par la médiane
        median_price = df[price_col].median()
        df[price_col] = df[price_col].fillna(median_price)
    else:
        df[price_col] = 0.0

    # 3. Construire un "Rich Text" pour le SentenceTransformer
    def create_rich_text(row):
        text = f"Product: {row[title_col]}. "
        if str(row[desc_col]).strip():
            # On limite la description à 800 caractères pour ne pas diluer le reste
            desc = str(row[desc_col]).strip()[:800] 
            text += f"Description: {desc}. "
        if row[brand_col] != 'Unknown':
            text += f"Brand: {row[brand_col]}. "
        if row[price_col] > 0:
            text += f"Price: {row[price_col]:.2f} USD."
        return text.strip()

    df['text'] = df.apply(create_rich_text, axis=1)

    # 4. Nettoyage léger (On GARDE la ponctuation !)
    def clean_text(s):
        s = re.sub(r'\s+', ' ', s)
        return s.strip()

    df['text'] = df['text'].apply(clean_text)

    # 5. Encodage Tabulaire pour le modèle Level 1 (LightGBM) si besoin
    label_enc = None
    if brand_col in df.columns:
        label_enc = LabelEncoder()
        df[brand_col + '_encoded'] = label_enc.fit_transform(df[brand_col].astype(str))

    # 6. Sélection des colonnes finales
    columns_to_keep = [id_col, 'text', price_col]
    
    # Ajout du Ground Truth s'il est présent
    if category_col in df.columns:
        columns_to_keep.append(category_col)
        
    if label_enc:
        columns_to_keep.append(brand_col + '_encoded')

    return df[columns_to_keep], label_enc

In [8]:
# Bien mettre les catégories dans la colonne "level_1_category" 

df_catalog_ground_truch = df_catalog.merge(
    df_ground_truth[['hashed_external_id', 'level_1_name']], 
    on='hashed_external_id', 
    how='left'
)


df_nlp, brand_encoder = preprocess_for_nlp(
    df_catalog_ground_truch,
    title_col='title',        
    desc_col='description',   
    brand_col='brand',
    id_col='hashed_external_id',
    price_col='sale_price',
    category_col='level_1_name' 
)

# Affichage de quelques exemples pour vérifier
for col in [1 , 10 , 100 , 500 , 999]:
    print(f"\n--- Product index {col} ---")
    print(df_nlp.iloc[col]['text'])
    # Si vous voulez aussi vérifier que la catégorie est bien là :
    # print("Catégorie L1 :", df_nlp.iloc[col]['level_1_category'])


--- Product index 1 ---
Product: Disney Villains Tarot Deck and Guidebook Movie Tarot Deck Pop Culture Tarot by Minerva Siegel. Description: Let Maleficent, Captain Hook, and other classic baddies guide your tarot practice with the only official tarot deck featuring Disney's most wicked villains.Disney's most iconic villains have taken over tarot in this dastardly take on a traditional 78-card deck. Featuring the notorious ne'er-do-wells from classic animated films like 101 Dalmations, The Little Mermaid, Sleeping Beauty, and more, this tarot deck reimagines Cruella de Vil, Ursula, Maleficent and the whole motley crew in original illustrations based on classic tarot iconography. Including both the Major and Minor Arcana, the set also comes with a helpful guidebook with explanations of each card's meaning, as well as simple spreads for easy readings. Packaged in a sturdy, decorative gift box, this devious deck of tarot cards is . Price: 24.99 USD.

--- Product index 10 ---
Product: We 

In [9]:
df_nlp

,hashed_external_id,text,sale_price,level_1_name,brand_encoded
0,-2772291400701920348,Product: The Hoodoo Tarot. Description: A divi...,35.00,religious & ceremonial,2416
1,-4184851053829790189,Product: Disney Villains Tarot Deck and Guideb...,24.99,religious & ceremonial,2416
2,-8778697834751578524,Product: Easy Tarot. Description: Created espe...,19.95,religious & ceremonial,2416
3,-3541475158234224984,Product: The Proudest Blue: A Story of Hijab a...,17.99,religious & ceremonial,2416
4,-2529310467283008815,Product: The Crystal Magic Tarot: Understand a...,24.95,religious & ceremonial,2416
...,...,...,...,...,...
128248,-731841779633851509,Product: Columbia Natural Feather and Down Sid...,77.70,home & garden,550
128249,-8372033233842851509,Product: Campania International Marlton Plante...,229.60,home & garden,432
128250,-5594512849126851501,Product: Beatrice Home Grand Hotel Waffle Knit...,38.08,home & garden,2694
128251,-402032100371851498,Product: Slickblue Snow Shovel with Wheels wit...,96.99,home & garden,1092


### check language distribution

In [ ]:
from langdetect import detect, DetectorFactory
# Pour rendre les résultats reproductibles
DetectorFactory.seed = 0

def detect_language(text):
    """
    Detect the language of a given text.
    
    Parameters
    ----------
    text : str
        Input text
    
    Returns
    -------
    lang_code : str
        ISO 639-1 language code (e.g., 'en' for English)
    """
    try:
        return detect(text)
    except:
        return "unknown"

language_df_level1_clean = df_nlp["text"].apply(detect_language)

In [ ]:
language_df_level1_clean.value_counts()

the vast majority of the text seems to be in English, with some other languages mixed in.

Decide to do nothing special for the non-English products, as they represent a small fraction of the data. We will keep them in the dataset and let the model learn from them as well.


# Fine tuning with L1 grounth truth

In [9]:
import os
import random
import warnings
import json
import time
import pandas as pd
import torch
import torch.nn.functional as F
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder 
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import TripletEvaluator
from torch.utils.data import DataLoader
from sentence_transformers import losses
from sentence_transformers.losses import BatchHardTripletLossDistanceFunction


# --- 0. CONFIGURATION DES HYPERPARAMÈTRES ---
HYPERPARAMS = {
    "model_name": "prdev/mini-gte",
    "batch_size": 8, 
    "epochs": 1,
    "max_seq_length": 128, 
    "sample_size": 20000,
    "distance_metric": "cosine", # 'cosine' ou 'euclidean'
    "warmup_steps": 100,
    "device": "mps" if torch.backends.mps.is_available() else "cpu"
}

# --- 1. CONFIGURATION SYSTÈME ---
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0" 
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

if HYPERPARAMS["device"] == "mps":
    torch.mps.empty_cache()
print(f"🚀 Appareil utilisé : {HYPERPARAMS['device']}")

# --- 2. PRÉPARATION DES DONNÉES ---
label_encoder = LabelEncoder()
df_nlp['label_l1_encoded'] = label_encoder.fit_transform(df_nlp['level_1_name'])

df_train_full, df_test = train_test_split(
    df_nlp, test_size=0.1, random_state=42, stratify=df_nlp['label_l1_encoded']
)

df_train = df_train_full.sample(min(HYPERPARAMS["sample_size"], len(df_train_full)), random_state=42)

train_examples = [
    InputExample(texts=[str(row['text']).strip()], label=int(row['label_l1_encoded'])) 
    for _, row in df_train.iterrows() if str(row['text']).strip()
]

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=HYPERPARAMS["batch_size"], drop_last=True)

🚀 Appareil utilisé : mps


In [ ]:
# --- 3. CONFIGURATION DU MODÈLE ET LOSS COSINUS ---
model = SentenceTransformer(HYPERPARAMS["model_name"], device=HYPERPARAMS["device"])
model.max_seq_length = HYPERPARAMS["max_seq_length"]

# On utilise l'énumération officielle de la librairie au lieu d'une fonction custom
if HYPERPARAMS["distance_metric"] == "cosine":
    distance_fct = BatchHardTripletLossDistanceFunction.cosine_distance
else:
    # Attention à l'orthographe "eucledian" propre à la librairie sentence-transformers
    distance_fct = BatchHardTripletLossDistanceFunction.eucledian 

train_loss = losses.BatchHardTripletLoss(model=model, distance_metric=distance_fct)
# --- 4. ÉVALUATEUR ---
def create_validation_triplets(df, num_triplets=300):
    anchors, positives, negatives = [], [], []
    grouped = df.groupby('level_1_name')['text'].apply(list).to_dict()
    categories = [c for c in grouped.keys() if len(grouped[c]) >= 2]
    for _ in range(num_triplets):
        target_cat = random.choice(categories)
        a, p = random.sample(grouped[target_cat], 2)
        neg_cat = random.choice([c for c in categories if c != target_cat])
        n = random.choice(grouped[neg_cat])
        anchors.append(str(a)); positives.append(str(p)); negatives.append(str(n))
    return anchors, positives, negatives

val_anchors, val_positives, val_negatives = create_validation_triplets(df_test)
evaluator = TripletEvaluator(val_anchors, val_positives, val_negatives, name='val-eval')

# --- 5. ENTRAÎNEMENT ET LOGGING ---
print(f"🔥 Début de l'entraînement avec Loss {HYPERPARAMS['distance_metric']}...")
start_time = time.time()

try:
    model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=HYPERPARAMS["epochs"],
        warmup_steps=HYPERPARAMS["warmup_steps"],
        evaluator=evaluator,
        evaluation_steps=500,
        output_path='./modele_checkpoint',
        save_best_model=True,
        show_progress_bar=True
    )
    
    print("Calcul du score final pour le rapport...")
    final_score = evaluator(model, output_path='./modele_checkpoint')
    
    duration = time.time() - start_time
    print(f"✅ Terminé en {duration:.2f}s")

    # --- 6. SAUVEGARDE DES RÉSULTATS DANS LE JSON ---
    log_file = "experiments_log.json"
    
    experiment_data = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "hyperparameters": HYPERPARAMS,
        "results": {
            "final_eval_score": final_score if final_score else "N/A",
            "training_duration_sec": duration
        }
    }

    if os.path.exists(log_file):
        with open(log_file, "r") as f:
            all_logs = json.load(f)
    else:
        all_logs = []

    all_logs.append(experiment_data)

    with open(log_file, "w") as f:
        json.dump(all_logs, f, indent=4)
        
    print(f"📊 Expérience enregistrée dans {log_file}")
    model.save('./modele_catalogue_finetuned_final')

except Exception as e:
    print(f"❌ Erreur : {e}")

# PRE / POST FINE TUNING EVALUATION

In [ ]:
import numpy as np
import json
import os
import torch
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
from sentence_transformers import SentenceTransformer

# --- Configuration du Device ---
device = HYPERPARAMS["device"]

# ==========================================
# 1. Fonction d'évaluation adaptée
# ==========================================
def evaluate_model(model_path, df_train_subset, df_test_subset, device):
    print(f"\n⏳ Évaluation du modèle : {model_path}")
    model = SentenceTransformer(model_path, device=device)
    model.max_seq_length = HYPERPARAMS["max_seq_length"]
    
    print("Génération des embeddings...")
    X_train = model.encode(df_train_subset['text'].tolist(), show_progress_bar=True, device=device)
    X_test = model.encode(df_test_subset['text'].tolist(), show_progress_bar=True, device=device)
    
    y_train = df_train_subset['label_l1_encoded'].values
    y_test = df_test_subset['label_l1_encoded'].values
    
    print("Calcul du k-NN (cosine)...")
    knn = KNeighborsClassifier(n_neighbors=5, metric=HYPERPARAMS['distance_metric'], weights='distance')
    knn.fit(X_train, y_train)
    
    y_pred = knn.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    return acc, y_test, y_pred

# ==========================================
# 2. Exécution des tests
# ==========================================
# On prend un échantillon du test (2000 produits) pour aller plus vite
df_eval_test = df_test.sample(min(2000, len(df_test)),random_state=42)

# --- Baseline ---
baseline_name = 'prdev/mini-gte'
acc_before, y_test_before, y_pred_before = evaluate_model(
    baseline_name, df_train, df_eval_test, device
)

# --- Fine-tuned ---
finetuned_path = './modele_catalogue_finetuned_final'
acc_after, y_test_after, y_pred_after = evaluate_model(
    finetuned_path, df_train, df_eval_test, device
)

# ==========================================
# 3. Mise à jour du JSON d'expérience
# ==========================================
log_file = "experiments_log.json"

if os.path.exists(log_file):
    with open(log_file, "r") as f:
        all_logs = json.load(f)
    
    # On met à jour la toute dernière expérience du JSON
    if len(all_logs) > 0:
        all_logs[-1]["results"].update({
            "accuracy_baseline": round(acc_before * 100, 2),
            "accuracy_finetuned": round(acc_after * 100, 2),
            "accuracy_gain": round((acc_after - acc_before) * 100, 2)
        })
        
        with open(log_file, "w") as f:
            json.dump(all_logs, f, indent=4)
        print(f"\n📊 Résultats d'évaluation ajoutés avec succès au fichier {log_file}")

# ==========================================
# 4. Affichage console
# ==========================================
print("\n" + "="*50)
print("🏆 BILAN COMPARATIF (DISTANCE COSINUS)")
print("="*50)
print(f"Précision Baseline   : {acc_before * 100:.2f} %")
print(f"Précision Fine-Tuned : {acc_after * 100:.2f} %")
print(f"Gain net             : {((acc_after - acc_before) * 100):+.2f} points")
print("="*50)

print("\n📊 Rapport détaillé (Fine-Tuned) :")
# Astuce pour éviter les crashs si certaines catégories manquent dans l'échantillon test
unique_classes = np.unique(np.concatenate((y_test_after, y_pred_after)))
target_names = label_encoder.inverse_transform(unique_classes)

print(classification_report(y_test_after, y_pred_after, labels=unique_classes, target_names=target_names))

# TEST XGBOOST VS LOGISTIC REGRESSION

In [16]:
import time
import json
import os
import numpy as np
import joblib  
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sentence_transformers import SentenceTransformer

print("⏳ Chargement du modèle Fine-Tuné pour générer les vecteurs...")
model_ft = SentenceTransformer('./modele_catalogue_finetuned_final', device=device)

# 1. Génération des vecteurs
print("Génération des embeddings Train (30k max)...")
X_train_vec = model_ft.encode(df_train['text'].tolist(), show_progress_bar=True, device=device)

print("Génération des embeddings Test (2k)...")
X_test_vec = model_ft.encode(df_eval_test['text'].tolist(), show_progress_bar=True, device=device)

y_train_vec = df_train['label_l1_encoded'].values
y_test_vec = df_eval_test['label_l1_encoded'].values

# ==========================================
# FIX XGBOOST : Recalibrage des labels (Boucher le "trou" de la classe 13)
# ==========================================
le_clf = LabelEncoder()
y_train_clf = le_clf.fit_transform(y_train_vec)

# Sécurité : On retire du test les produits dont la catégorie n'a jamais été vue dans le train
mask = np.isin(y_test_vec, le_clf.classes_)
X_test_vec = X_test_vec[mask]
y_test_vec = y_test_vec[mask]

# On transforme le test avec le nouvel encodeur sans trous
y_test_clf = le_clf.transform(y_test_vec)

# ==========================================
# TEST 1 : Régression Logistique 
# ==========================================
print("\n🚀 Entraînement Régression Logistique...")
start_lr = time.time()
clf_lr = LogisticRegression(max_iter=1000, n_jobs=-1)
clf_lr.fit(X_train_vec, y_train_clf) # On utilise les labels recalibrés
y_pred_lr_clf = clf_lr.predict(X_test_vec)

# On ramène les prédictions dans leur format d'origine (avec le "13" vide)
y_pred_lr = le_clf.inverse_transform(y_pred_lr_clf)
acc_lr = accuracy_score(y_test_vec, y_pred_lr)
time_lr = time.time() - start_lr
print(f"✅ Terminé en {time_lr:.2f}s | Précision : {acc_lr * 100:.2f}%")

# ==========================================
# TEST 2 : XGBoost
# ==========================================
print("\n🌲 Entraînement XGBoost...")
start_xgb = time.time()
clf_xgb = XGBClassifier(n_estimators=100, learning_rate=0.1, tree_method='hist', n_jobs=-1)
clf_xgb.fit(X_train_vec, y_train_clf) # XGBoost est maintenant content !
y_pred_xgb_clf = clf_xgb.predict(X_test_vec)

# On ramène les prédictions
y_pred_xgb = le_clf.inverse_transform(y_pred_xgb_clf)
acc_xgb = accuracy_score(y_test_vec, y_pred_xgb)
time_xgb = time.time() - start_xgb
print(f"✅ Terminé en {time_xgb:.2f}s | Précision : {acc_xgb * 100:.2f}%")

# ==========================================
# COMPARAISON FINALE
# ==========================================
best_acc = max(acc_lr, acc_xgb)
best_model_name = "LogReg" if acc_lr > acc_xgb else "XGBoost"

print("\n" + "="*50)
print("🏆 COMPARAISON DES CLASSIFIEURS (LEVEL 1)")
print("="*50)
knn_score_str = f"{acc_after * 100:.2f}%" if 'acc_after' in locals() else "Non calculé"
print(f"Ancien score (k-NN)           : {knn_score_str}") 
print(f"Nouveau score (LogReg)        : {acc_lr * 100:.2f}%")
print(f"Nouveau score (XGBoost)       : {acc_xgb * 100:.2f}%")
print("="*50)

# ==========================================
# MISE À JOUR DU JSON
# ==========================================
log_file = "experiments_log.json"

if os.path.exists(log_file):
    with open(log_file, "r") as f:
        all_logs = json.load(f)
    
    if len(all_logs) > 0:
        all_logs[-1]["results"].update({
            "accuracy_logreg": round(acc_lr * 100, 2),
            "accuracy_xgboost": round(acc_xgb * 100, 2),
            "best_classifier": best_model_name,
            "best_classifier_accuracy": round(best_acc * 100, 2),
            "train_time_logreg_sec": round(time_lr, 2),
            "train_time_xgboost_sec": round(time_xgb, 2)
        })
        
        with open(log_file, "w") as f:
            json.dump(all_logs, f, indent=4)
        print(f"\n📊 Résultats des classifieurs ajoutés avec succès au fichier {log_file}")

# ==========================================
# RAPPORT DÉTAILLÉ DU GAGNANT
# ==========================================
best_pred = y_pred_lr if acc_lr > acc_xgb else y_pred_xgb
print(f"\n📊 Rapport détaillé du meilleur modèle ({best_model_name}) :")
unique_classes = np.unique(np.concatenate((y_test_vec, best_pred)))
target_names = label_encoder.inverse_transform(unique_classes)
print(classification_report(y_test_vec, best_pred, labels=unique_classes, target_names=target_names))

# ==========================================
# 💾 SAUVEGARDE DU MODÈLE ET DES ENCODEURS
# ==========================================
save_dir = './saved_models'
os.makedirs(save_dir, exist_ok=True)

# 1. On sauvegarde le meilleur classifieur
best_clf = clf_lr if best_model_name == "LogReg" else clf_xgb
joblib.dump(best_clf, os.path.join(save_dir, 'best_l1_classifier.joblib'))

# 2. On sauvegarde le LabelEncoder de recalibrage (le_clf)
joblib.dump(le_clf, os.path.join(save_dir, 'label_encoder_recalibrage.joblib'))

# 3. On sauvegarde le LabelEncoder principal (label_encoder) défini plus haut dans votre notebook
joblib.dump(label_encoder, os.path.join(save_dir, 'label_encoder_principal.joblib'))

print(f"\n💾 Modèle ({best_model_name}) et Encodeurs sauvegardés avec succès dans le dossier '{save_dir}' !")

⏳ Chargement du modèle Fine-Tuné pour générer les vecteurs...
Génération des embeddings Train (30k max)...


Batches:   0%|          | 0/625 [00:00<?, ?it/s]

Génération des embeddings Test (2k)...


Batches:   0%|          | 0/63 [00:00<?, ?it/s]


🚀 Entraînement Régression Logistique...
✅ Terminé en 14.12s | Précision : 91.15%

🌲 Entraînement XGBoost...
✅ Terminé en 101.57s | Précision : 91.30%

🏆 COMPARAISON DES CLASSIFIEURS (LEVEL 1)
Ancien score (k-NN)           : Non calculé
Nouveau score (LogReg)        : 91.15%
Nouveau score (XGBoost)       : 91.30%

📊 Résultats des classifieurs ajoutés avec succès au fichier experiments_log.json

📊 Rapport détaillé du meilleur modèle (XGBoost) :
                           precision    recall  f1-score   support

   animals & pet supplies       0.80      0.83      0.82        24
    apparel & accessories       0.98      0.97      0.98       319
     arts & entertainment       0.84      0.68      0.75        68
           baby & toddler       0.91      0.54      0.68        39
    business & industrial       0.00      0.00      0.00         2
         cameras & optics       0.86      0.98      0.92        55
              electronics       0.92      0.85      0.88       132
food, beverages

# Extract all level 2 categories from the training set

In [20]:
import os

# Nom du fichier source
fichier_taxonomie = 'taxonomy.txt'

level_2_categories = set()

# 1. Lecture du fichier et extraction des Level 2
with open(fichier_taxonomie, 'r', encoding='utf-8') as f:
    for line in f:
        # On sépare les identifiants (gauche) des noms (droite) via la tabulation (\t)
        parts = line.strip().split('\t')
        
        if len(parts) == 2:
            nom_complet = parts[1]
            # On sépare les niveaux par " > "
            niveaux = nom_complet.split(' > ')
            
            # Si le chemin a au moins 2 niveaux, on prend le deuxième (index 1)
            if len(niveaux) >= 2:
                level_2_categories.add(niveaux[1].strip())

# On convertit le set en liste triée par ordre alphabétique
level_2_liste = sorted(list(level_2_categories))

print(f"✅ Succès ! {len(level_2_liste)} catégories de Level 2 uniques trouvées.")

# 2. Sauvegarde de la liste complète dans un fichier texte simple
with open('liste_level_2_complete.txt', 'w', encoding='utf-8') as f:
    for cat in level_2_liste:
        f.write(cat + '\n')

# 3. Préparation pour les requêtes (100 mots par 100 mots)
chunk_size = 100
chunks = [level_2_liste[i:i + chunk_size] for i in range(0, len(level_2_liste), chunk_size)]

print("\n📦 Découpage en lots :")
for i, chunk in enumerate(chunks):
    print(f"  - Lot n°{i+1} : {len(chunk)} catégories")
    
    # Optionnel : Sauvegarder chaque lot dans un fichier CSV séparé
    nom_fichier_chunk = f'level_2_lot_{i+1}.csv'
    with open(nom_fichier_chunk, 'w', encoding='utf-8') as f_chunk:
        f_chunk.write("Level 2;Mots proches\n") # En-tête
        for cat in chunk:
            f_chunk.write(f"{cat};\n")

print("\n🚀 Fichiers générés avec succès ! Vous êtes prêt pour la suite.")

✅ Succès ! 191 catégories de Level 2 uniques trouvées.

📦 Découpage en lots :
  - Lot n°1 : 100 catégories
  - Lot n°2 : 91 catégories

🚀 Fichiers générés avec succès ! Vous êtes prêt pour la suite.


# Level 2 categories distribution

In [12]:
import pandas as pd
import networkx as nx
import numpy as np
import torch
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import joblib


# ==========================================
# 0. CHARGEMENT DES DEUX MODÈLES
# ==========================================
device = "mps" if torch.backends.mps.is_available() else "cpu"

print("⏳ Chargement du modèle Fine-Tuné (Spécialiste Level 1)...")
model_ft = SentenceTransformer('./modele_catalogue_finetuned_final', device=device)

print("⏳ Chargement du modèle STS (Spécialiste Level 2 / Zero-Shot)...")
# BAAI/bge-small-en-v1.5 est actuellement l'un des meilleurs modèles STS légers et rapides
model_sts = SentenceTransformer('BAAI/bge-small-en-v1.5', device=device)

# ==========================================
# 1. PRÉ-CALCUL DES VECTEURS LEVEL 2 (AVEC LE MODÈLE STS)
# ==========================================
df_l2_desc = pd.read_csv('categories_level_2.csv', sep=';')
l2_descriptions_dict = dict(zip(df_l2_desc['Level 2'], df_l2_desc['English Keywords']))

print("⏳ Pré-calcul des embeddings Level 2 avec le modèle STS...")
l2_embeddings_sts = {}

level_1_nodes = list(G.successors(root))

for l1 in level_1_nodes:
    level_2_nodes = list(G.successors(l1))
    
    for l2 in level_2_nodes:
        if l2 not in l2_embeddings_sts:
            texte_a_embedder = l2_descriptions_dict.get(l2, l2) 
            
            # ATTENTION ICI : On utilise le modèle STS généraliste, pas le fine-tuné !
            vec = model_sts.encode([texte_a_embedder], device=device)
            l2_embeddings_sts[l2] = vec

print(f"✅ {len(l2_embeddings_sts)} sous-catégories vectorisées pour le Zero-Shot !")

# ==========================================
# 2. LE PIPELINE DE PRÉDICTION (MISE À JOUR)
# ==========================================
def predict_full_category_dual_model(product_text, model_level1, model_level2, 
                                     l1_classifier, l1_label_encoder, l1_recalibrator, graph):
    """
    Pipeline mis à jour pour prendre en compte le classifieur chargé et le double décodage.
    """
    # --- ETAPE A : CLASSIFICATION LEVEL 1 ---
    vec_l1 = model_level1.encode([product_text], device=device)
    
    # 1. Prédiction brute (donne l'ID recalibré)
    l1_encoded_clf_pred = l1_classifier.predict(vec_l1)
    # 2. On repasse à l'ID original
    l1_encoded_original = l1_recalibrator.inverse_transform(l1_encoded_clf_pred)
    # 3. On récupère le texte (Le nom de la catégorie Level 1)
    predicted_l1 = l1_label_encoder.inverse_transform(l1_encoded_original)[0]
    
    # --- ETAPE B : RÉCUPÉRATION DES ENFANTS ---
    if predicted_l1 in graph:
        possible_l2_cats = list(graph.successors(predicted_l1))
    else:
        possible_l2_cats = []
    
    if not possible_l2_cats:
        return predicted_l1, "Aucune sous-catégorie", 0.0
    
    # --- ETAPE C : RETRIEVAL LEVEL 2 (ZERO-SHOT) ---
    vec_l2_product = model_level2.encode([product_text], device=device)
    
    best_l2 = None
    best_score = -1
    
    for l2_cat in possible_l2_cats:
        l2_vec_reference = l2_embeddings_sts.get(l2_cat)
        
        if l2_vec_reference is not None:
            score = cosine_similarity(vec_l2_product, l2_vec_reference)[0][0]
            if score > best_score:
                best_score = score
                best_l2 = l2_cat
                
    return predicted_l1, best_l2, best_score

# ==========================================
# 3. TEST DU PIPELINE AVEC CHARGEMENT
# ==========================================
print("\n⏳ Chargement du classifieur et des encodeurs...")
save_dir = './saved_models'

loaded_clf = joblib.load(os.path.join(save_dir, 'best_l1_classifier.joblib'))
loaded_label_encoder = joblib.load(os.path.join(save_dir, 'label_encoder_principal.joblib'))
loaded_le_clf = joblib.load(os.path.join(save_dir, 'label_encoder_recalibrage.joblib'))

test_products = [
    "Sony PlayStation 5 Console with wireless controller and 1TB SSD",
    "Royal Canin Dry Dog Food for adult golden retrievers, 30 lbs",
    "Minimalist wooden coffee table with oak finish for living room",
    "Waterproof hiking boots for men, size 10, breathable material"
]

print("\n" + "="*60)
print("🎯 TEST DU PIPELINE (ARCHITECTURE DOUBLE MODÈLE)")
print("="*60)

for text in test_products:
    l1, l2, score = predict_full_category_dual_model(
        product_text=text, 
        model_level1=model_ft,    
        model_level2=model_sts,   
        l1_classifier=loaded_clf,                   
        l1_label_encoder=loaded_label_encoder,      
        l1_recalibrator=loaded_le_clf,              
        graph=G
    )
    
    print(f"📦 Produit : {text[:50]}...")
    print(f"   ┣ Level 1 prédit : {l1}")
    print(f"   ┗ Level 2 prédit : {l2} (Confiance STS : {score:.2f})\n")

⏳ Chargement du modèle Fine-Tuné (Spécialiste Level 1)...
⏳ Chargement du modèle STS (Spécialiste Level 2 / Zero-Shot)...
⏳ Pré-calcul des embeddings Level 2 avec le modèle STS...
✅ 191 sous-catégories vectorisées pour le Zero-Shot !

⏳ Chargement du classifieur et des encodeurs...

🎯 TEST DU PIPELINE (ARCHITECTURE DOUBLE MODÈLE)
📦 Produit : Sony PlayStation 5 Console with wireless controlle...
   ┣ Level 1 prédit : electronics
   ┗ Level 2 prédit : video game console accessories (Confiance STS : 0.68)

📦 Produit : Royal Canin Dry Dog Food for adult golden retrieve...
   ┣ Level 1 prédit : animals & pet supplies
   ┗ Level 2 prédit : pet supplies (Confiance STS : 0.55)

📦 Produit : Minimalist wooden coffee table with oak finish for...
   ┣ Level 1 prédit : furniture
   ┗ Level 2 prédit : tables (Confiance STS : 0.69)

📦 Produit : Waterproof hiking boots for men, size 10, breathab...
   ┣ Level 1 prédit : apparel & accessories
   ┗ Level 2 prédit : shoes (Confiance STS : 0.65)



In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

def batch_predict_and_export(df, model_level1, model_level2, 
                             l1_classifier, l1_label_encoder, l1_recalibrator, 
                             graph, embeddings_dict, batch_size=64):
    """
    Pipeline optimisé pour traiter des milliers de produits d'un coup et exporter les résultats.
    """
    # 1. Extraction des listes pour le traitement rapide
    # Remplacez 'text' par le nom de la colonne qui contient la description de vos produits
    texts = df['text'].tolist() 
    hashes = df['hashed_external_id'].tolist()
    
    # --- ÉTAPE 1 : ENCODAGE & PRÉDICTION LEVEL 1 EN LOTS ---
    print(f"⏳ 1/4: Encodage Level 1 pour {len(texts)} produits (Batching)...")
    l1_vectors = model_level1.encode(texts, batch_size=batch_size, show_progress_bar=True, device=device)
    
    print("⏳ 2/4: Prédiction Level 1 par le classifieur (XGBoost/LogReg)...")
    l1_clf_preds = l1_classifier.predict(l1_vectors)
    l1_original_encoded = l1_recalibrator.inverse_transform(l1_clf_preds)
    l1_labels = l1_label_encoder.inverse_transform(l1_original_encoded)
    
    # --- ÉTAPE 2 : ENCODAGE & RETRIEVAL LEVEL 2 EN LOTS ---
    print(f"⏳ 3/4: Encodage Level 2 (Modèle STS) pour {len(texts)} produits...")
    l2_vectors = model_level2.encode(texts, batch_size=batch_size, show_progress_bar=True, device=device)
    
    print("⏳ 4/4: Calcul rapide des similarités Cosinus pour le Level 2...")
    l2_labels = []
    l2_scores = []
    
    # Ici on boucle, mais sur des calculs purement mathématiques très rapides
    for i in tqdm(range(len(texts))):
        predicted_l1 = l1_labels[i]
        
        # Le vecteur du produit (reshape pour Scikit-Learn)
        product_l2_vec = l2_vectors[i].reshape(1, -1)
        
        # Récupération des enfants possibles dans la taxonomie
        possible_l2_cats = list(graph.successors(predicted_l1)) if predicted_l1 in graph else []
        
        if not possible_l2_cats:
            l2_labels.append("Pas de sous-catégorie")
            l2_scores.append(0.0)
            continue
            
        best_l2 = None
        best_score = -1
        
        for l2_cat in possible_l2_cats:
            l2_vec_reference = embeddings_dict.get(l2_cat)
            
            if l2_vec_reference is not None:
                ref_vec = l2_vec_reference.reshape(1, -1) if len(l2_vec_reference.shape) == 1 else l2_vec_reference
                score = cosine_similarity(product_l2_vec, ref_vec)[0][0]
                
                if score > best_score:
                    best_score = score
                    best_l2 = l2_cat
                    
        l2_labels.append(best_l2)
        l2_scores.append(best_score)
        
    # --- ÉTAPE 3 : CRÉATION DU DATAFRAME ET EXPORT ---
    print("\n📦 Création du fichier de résultats...")
    df_results = pd.DataFrame({
        'external_hash': hashes,
        'predicted_level_1': l1_labels,
        'predicted_level_2': l2_labels,
        'sts_confidence_score': l2_scores # Utile pour filtrer ceux où le modèle n'est pas sûr !
    })
    
    return df_results


# ==========================================
# 🚀 LANCEMENT SUR VOS DONNÉES GLOBALES
# ==========================================

# Remplacer df_full par votre dataframe complet contenant vos données à prédire
# Exemple: df_full = pd.read_csv("mon_catalogue_complet.csv")

df_predictions = batch_predict_and_export(
    df=df_nlp, 
    model_level1=model_ft,
    model_level2=model_sts,
    l1_classifier=loaded_clf,             # Votre modèle sauvegardé
    l1_label_encoder=loaded_label_encoder,# Votre encodeur sauvegardé
    l1_recalibrator=loaded_le_clf,        # Votre encodeur de correction sauvegardé
    graph=G,
    embeddings_dict=l2_embeddings_sts,
    batch_size=64 
)

# Exportation en CSV final
fichier_export = "predictions_catalogue_complet.csv"
df_predictions.to_csv(fichier_export, index=False, sep=';', encoding='utf-8')

print(f"\n✅ TERMINÉ ! Le fichier '{fichier_export}' a été généré avec succès.")
print(df_predictions.head())

⏳ 1/4: Encodage Level 1 pour 128253 produits (Batching)...


Batches:   0%|          | 0/2004 [00:00<?, ?it/s]

⏳ 2/4: Prédiction Level 1 par le classifieur (XGBoost/LogReg)...
⏳ 3/4: Encodage Level 2 (Modèle STS) pour 128253 produits...


Batches:   0%|          | 0/2004 [00:00<?, ?it/s]